In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import ViTModel
from torch.utils.data import Dataset,DataLoader,TensorDataset
import json
import cv2
import numpy as np
from scipy.spatial.transform import Rotation as R
from tqdm import tqdm

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
in_channels=3
emb_dim=512
img_size=224
patch_size=16
num_patches=(img_size//patch_size)**2
num_heads=4
num_blocks=2
out_dim=9

In [ ]:
def Position_Embedding(num_patches,embed_dim):
  pe=torch.zeros(num_patches+1,embed_dim)
  position=torch.arange(0,num_patches+1).unsqueeze(1)

  i=torch.arange(0,embed_dim//2)

  freq=1/(10000**(2*i/embed_dim))

  angles=position*freq
  sin=torch.sin(angles)
  cos=torch.sin(angles)
  pe[:,0::2]=sin
  pe[:,1::2]=cos

  return pe.unsqueeze(0).to(device)

In [ ]:
Position_Embedding(5,1024).shape

torch.Size([1, 6, 1024])

In [ ]:
class PatchEmbedding(nn.Module):
  def __init__(self):
    super().__init__()
    self.proj=nn.Conv2d(in_channels=in_channels,out_channels=emb_dim,kernel_size=patch_size,stride=patch_size)
  def forward(self,x):
    out=self.proj(x)
    out=out.flatten(2)
    out=out.permute(0,2,1)
    return out


In [ ]:
class Block(nn.Module):
  def __init__(self):
    super().__init__()
    self.attn=nn.MultiheadAttention(embed_dim=emb_dim,num_heads=num_heads,batch_first=True)
    self.ff=nn.Sequential(
        nn.Linear(emb_dim,4*emb_dim),
        nn.GELU(),
        nn.Linear(4*emb_dim,emb_dim)
    )
    self.ln1=nn.LayerNorm(emb_dim)
    self.ln2=nn.LayerNorm(emb_dim)
  def forward(self,x):
    x=self.ln1(x)
    x=x+self.attn(x,x,x)[0]
    x=self.ln2(x)
    x=x+self.ff(x)
    return x


In [ ]:
class Vit(nn.Module):
  def __init__(self):
    super().__init__()
    self.patch=PatchEmbedding()
    self.cls=nn.Parameter(torch.randn(1,1,emb_dim))
    self.pos_emb=Position_Embedding(num_patches,emb_dim)
    self.blocks=nn.ModuleList(Block() for _ in range(num_blocks))
    self.final=nn.Sequential(
        nn.LayerNorm(emb_dim),
        nn.Linear(emb_dim,out_dim)
    )
  def forward(self,x):
    x=self.patch(x)
    B,T,C=x.shape
    cls_token=self.cls.expand(B,-1,-1)
    x=torch.cat((cls_token,x),dim=1)

    x=x+self.pos_emb
    for block in self.blocks:
      x=block(x)
    x=x[:,0]
    x=self.final(x)
    return x


In [ ]:
model=Vit().to(device)

In [ ]:
# model.load_state_dict(torch.load('model_5.pt'))

<All keys matched successfully>

# Utils

In [ ]:

def get_target(q,T,x_min,y_min,w,h,K):
    bx=x_min+w/2
    by=y_min+h/2

    q_scipy=[q[1],q[2],q[3],q[0]]
    R_mat=R.from_quat(q_scipy).as_matrix()


    fx,fy=K[0,0],K[1,1]
    cx,cy=K[0,2],K[1,2]

    X,Y,Z=T

    x=fx*X/Z+cx
    y=fy*Y/Z+cy

    Ux=(x-bx)/w
    Uy=(y-by)/h

    W,H=1920,1200

    alpha=1.6

    sx=W/(alpha*w)
    sy=H/(h)

    Uz=0.5*(1/sx+1/sy)*Z


    T_vec=np.array([X,Y,Z])
    T_hat=T_vec/(np.linalg.norm(T_vec)+1e-8)

    ez=np.array([0,0,1])

    u=np.cross(T_hat,ez)

    norm_u=np.linalg.norm(u)

    if norm_u<1e-6:
        delta_R=np.eye(3)

    else:
        u=u/norm_u
        theta=np.arccos(np.clip(np.dot(T_hat,ez),-1,1))

        skew_symm_m=np.array([
            [0, -u[2], u[1]],
            [u[2], 0, -u[0]],
            [-u[1], u[0], 0]
        ])

        delta_R=np.eye(3)+np.sin(theta)*skew_symm_m+(1-np.cos(theta))*(skew_symm_m@skew_symm_m)

    R_prime=delta_R@R_mat

    r1_hat=R_prime[:,0]
    r2_hat=R_prime[:,1]

    target=np.concatenate([[Ux,Uy,Uz],r1_hat,r2_hat])
    R_prime=R_prime.flatten()

    return target,T_vec,R_mat


def get_inference(pred,bbox,K):
    K=torch.tensor(K,dtype=torch.float32,device=pred.device)
    bbox=bbox.to(pred.device)
    x_min=bbox[:,0]
    y_min=bbox[:,1]
    w=bbox[:,2]
    h=bbox[:,3]

    bx=x_min+w/2
    by=y_min+h/2

    fx,fy=K[0,0],K[1,1]
    cx,cy=K[0,2],K[1,2]

    W,H=1920,1200

    alpha=1.6

    Ux=pred[:,0]
    Uy=pred[:,1]
    Uz=pred[:,2]

    r1_hat=pred[:,3:6]
    r2_hat=pred[:,6:9]

    sx=W/(alpha*w)
    sy=H/h

    Z=Uz/(0.5*(1/sx+1/sy))

    X=(bx+Ux*w-cx)*Z/fx
    Y=(by+Uy*h-cy)*Z/fy

    r1=nn.functional.normalize(r1_hat,dim=1)
    r2=r2_hat-torch.sum(r2_hat*r1,dim=1,keepdim=True)*r1
    r2=nn.functional.normalize(r2,dim=1)
    r3=torch.cross(r1,r2,dim=1)
    R_prime=torch.stack([r1,r2,r3],axis=2)


    T_vec=torch.stack([X,Y,Z],axis=1)
    T_hat=nn.functional.normalize(T_vec,dim=1)

    ez=torch.tensor([0,0,1],dtype=torch.float32,device=pred.device).expand(pred.shape[0],3)

    u=torch.cross(T_hat,ez,dim=1)

    u=nn.functional.normalize(u,dim=1)


    theta=torch.acos(torch.clamp(torch.sum(T_hat*ez,dim=1),-1,1))

    ux,uy,uz=u[:,0],u[:,1],u[:,2]

    zero=torch.zeros_like(ux)
    skew_symm_m=torch.stack([
        torch.stack([zero,-uz,uy],dim=1),
        torch.stack([uz,zero,-ux],dim=1),
        torch.stack([-uy,ux,zero],dim=1)
    ],dim=1)

    sin_theta=torch.sin(theta).unsqueeze(-1).unsqueeze(-1)
    cos_theta=torch.cos(theta).unsqueeze(-1).unsqueeze(-1)

    delta_R=torch.eye(3,device=pred.device).unsqueeze(0).expand(pred.shape[0],3,3)+sin_theta*skew_symm_m+(1-cos_theta)*torch.bmm(skew_symm_m,skew_symm_m)
    R_pred=torch.bmm(delta_R.transpose(1,2),R_prime)
    return T_vec,R_pred





    # delta_R=np.eye(3)+np.sin(theta)*skew_symm_m+(1-np.cos(theta))*(skew_symm_m@skew_symm_m)

    # R_pred=delta_R.T@R_prime

    # return T_vec,R_pred






# Dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("isidorotamassia/speed")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/isidorotamassia/speed/versions/1


In [ ]:
K=np.array([
    [2988.5795163815555, 0, 960],
    [0, 2988.3401159176124, 600],
    [0, 0, 1]
])



class PoseDatatset(Dataset):
  def __init__(self,json_path,img_dir):
    with open(json_path,'r') as f:
      self.data=json.load(f)
    self.img_dir=img_dir
  def __len__(self):
    return len(self.data)

  def __getitem__(self,idx):
    item=self.data[idx]
    img_path=self.img_dir+"/"+item['filename']
    img=cv2.imread(img_path)

    img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

    x_min=int(item['x_min'])
    y_min=int(item['y_min'])
    w=int(item['w'])
    h=int(item['h'])

    crop_img=img[y_min:y_min+h,x_min:x_min+w]

    crop_img=cv2.resize(crop_img,(224,224))
    crop_img=crop_img.astype(np.float32)
    crop_img=crop_img/255.0
    crop_img=torch.from_numpy(crop_img)
    crop_img=crop_img.permute(2,0,1)



    target,T,R_prime=get_target(item['q'],item['T'],x_min,y_min,w,h,K)
    crop_img=crop_img.to(torch.float32)
    target=torch.tensor(target,dtype=torch.float32)
    bbox=torch.tensor([x_min,y_min,w,h],dtype=torch.float32)
    T=torch.tensor(T,dtype=torch.float32)
    R_prime=torch.tensor(R_prime,dtype=torch.float32)


    return crop_img,target,T,R_prime,bbox



In [ ]:
train_ds=PoseDatatset("pose_train.json",path+"/speedplusv2/synthetic/images")
train_loader=DataLoader(train_ds,batch_size=16,shuffle=True)

In [ ]:
val_ds=PoseDatatset("pose_val.json",path+"/speedplusv2/synthetic/images")
val_loader=DataLoader(val_ds,batch_size=16,shuffle=True)

In [ ]:
crit=nn.MSELoss()
optimizer=optim.Adam(model.parameters(),lr=1e-4)
num_epochs=50

In [ ]:
def translation_error(T_pred,T_gt):
  return torch.norm(T_pred-T_gt,dim=1).mean()

In [ ]:
def rotation_error(R_pred,R_gt):
  R_gt=R_gt.view(-1,3,3)
  R_mul=torch.bmm(R_pred.transpose(1,2),R_gt)
  trace=R_mul[:,0,0]+R_mul[:,1,1]+R_mul[:,2,2]
  cos_theta=(trace-1)/2
  cos_theta=torch.clamp(cos_theta,-1,1)
  theta=torch.acos(cos_theta)
  theta_deg=torch.rad2deg(theta)
  theta_error=torch.mean(theta_deg)

  return theta_error

In [ ]:
def check_val():
  model.eval()
  print("Evaluaion Begins")
  t_error=0
  r_error=0
  for x,y,T,R_prime,bbox in tqdm(val_loader):
    x=x.to(device)
    y=y.to(device)
    T=T.to(device)
    R_prime=R_prime.to(device)
    bbox=bbox.to(device)
    T_pred,R_prime_pred=get_inference(model(x),bbox,K)
    t_error+=translation_error(T_pred,T).item()
    r_error+=rotation_error(R_prime_pred,R_prime).item()
  print(f"Translation Error: {t_error/len(val_loader)}")
  print(f"Rotation Error: {r_error/len(val_loader)}")
  model.train()


In [ ]:
for epochs in range(num_epochs):
  mean_loss=0
  for x,y,_,_,_ in tqdm(train_loader):
    x=x.to(device)
    y=y.to(device)
    pred=model(x)
    loss=crit(pred,y)
    mean_loss+=loss.item()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
  print(f"Epoch: {epochs+1} Loss: {mean_loss/len(train_loader)}")

  if (epochs+1)%5==0:
    check_val()
    torch.save(model.state_dict(),f"model_{epochs+1}.pt")



100%|██████████| 2998/2998 [12:52<00:00,  3.88it/s]


Epoch: 1 Loss: 0.15036440014048885


100%|██████████| 2998/2998 [12:46<00:00,  3.91it/s]


Epoch: 2 Loss: 0.13975202706111242


  8%|▊         | 236/2998 [00:59<11:30,  4.00it/s]


KeyboardInterrupt: 

In [ ]:
check_val()
torch.save(model.state_dict(),f"model_{epochs}.pt")

Evaluaion Begins


100%|██████████| 750/750 [03:12<00:00,  3.89it/s]


Translation Error: 0.4098546038269997
Rotation Error: 65.94747793070475
